# Daphnet FoG Detection — Improved Pipeline
## Part 1: Data Loading, Preprocessing & Feature Extraction

Fixes ALL identified issues from diagnostic analysis:
1. Applies 4th-order Butterworth bandpass filter (0.5-20 Hz)
2. Filters out annotation=0 (non-experiment) samples
3. Uses 4-second windows (256 samples @ 64 Hz) per literature
4. Welch nperseg capped at window length
5. Handles subjects with zero FoG in LOSO metrics
6. Majority voting (>50%) for window labelling
7. Feature selection via mutual information (k=60)
8. Threshold optimization via Youden's J

In [ ]:
from __future__ import annotations

import sys, os, time, json, warnings, logging
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
from scipy.optimize import minimize
from joblib import Parallel, delayed
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

# ── Project path ──
PROJECT_ROOT = Path(os.getcwd()).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from loaders import DaphnetDatasetLoader
from features.extractors import FeatureExtractor
from utils.pipeline_utils import (
    interpolate_missing, bandpass_filter, zscore_normalize, create_windows,
    youden_threshold, compute_metrics, get_classifiers, get_param_grids,
    prepare_fold, preprocess_features, train_and_evaluate_classifier,
    build_base_model, aggregate_results, print_results_table, print_fusion_results,
    HAS_XGB, HAS_SMOTE,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("daphnet")

## Constants

In [ ]:
FS = 64
WINDOW_SEC = 4.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)  # 256
TRAIN_OVERLAP = 0.50
TEST_OVERLAP = 0.0
LABEL_THRESH = 0.50  # majority voting
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 20.0, 4
NPERSEG = min(128, WINDOW_SAMPLES)
K_FEATURES = 60
SEED = 42
N_INNER_CV = 3
N_SEARCH_ITER = 20
ZERO_FOG_SUBJECTS = {4, 10}

SENSOR_COLS = {
    "ankle": ["acc_forward_ankle", "acc_vertical_ankle", "acc_lateral_ankle"],
    "thigh": ["acc_forward_thigh", "acc_vertical_thigh", "acc_lateral_thigh"],
    "trunk": ["acc_forward_trunk", "acc_vertical_trunk", "acc_lateral_trunk"],
}
ALL_ACC = [c for cols in SENSOR_COLS.values() for c in cols]

DATASET_PATH = PROJECT_ROOT / "Datasets" / "Daphnet fog" / "dataset"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "daphnet_improved_results"

## Data Loading & Preprocessing

In [ ]:
def load_and_preprocess() -> Dict[int, Dict]:
    """Load Daphnet data, preprocess per subject/trial. Returns dict[subject_id] -> {windows, labels}."""
    log.info("Loading Daphnet dataset from %s", DATASET_PATH)
    loader = DaphnetDatasetLoader(str(DATASET_PATH))
    df = loader.load_all_data(verbose=False)

    # Filter out annotation=0 (non-experiment)
    n_before = len(df)
    df = df[df["annotation"] != 0].reset_index(drop=True)
    log.info("Filtered annotation=0: %d -> %d samples", n_before, len(df))

    # Binary labels: annotation 2 -> FoG (1), annotation 1 -> No FoG (0)
    df["fog_label"] = (df["annotation"] == 2).astype(int)

    subjects_data = {}
    for sid in sorted(df["subject_id"].unique()):
        sub_df = df[df["subject_id"] == sid]
        all_windows, all_labels = [], []

        for rid in sorted(sub_df["run_id"].unique()):
            trial = sub_df[sub_df["run_id"] == rid]
            signal = trial[ALL_ACC].values.astype(np.float64)
            labels_raw = trial["fog_label"].values.astype(np.float64)

            # Interpolate missing values before filtering
            signal = interpolate_missing(signal)
            labels_raw = np.nan_to_num(labels_raw, nan=0.0)

            # Bandpass filter
            signal = bandpass_filter(signal, FS, BP_LOW, BP_HIGH, BP_ORDER)

            # Add magnitude per sensor
            mags = []
            for sensor_name, cols in SENSOR_COLS.items():
                idxs = [ALL_ACC.index(c) for c in cols]
                mag = np.sqrt(np.sum(signal[:, idxs] ** 2, axis=1, keepdims=True))
                mags.append(mag)
            signal = np.hstack([signal, np.hstack(mags)])  # 9 + 3 = 12 channels

            # Z-score normalize per trial
            signal = zscore_normalize(signal)

            # Create windows
            w, l = create_windows(signal, labels_raw, WINDOW_SAMPLES, TRAIN_OVERLAP, LABEL_THRESH)
            if len(w) > 0:
                all_windows.append(w)
                all_labels.append(l)

        if all_windows:
            subjects_data[sid] = {
                "windows": np.concatenate(all_windows, axis=0),
                "labels": np.concatenate(all_labels, axis=0),
            }
            n_fog = int(np.sum(subjects_data[sid]["labels"]))
            log.info("  Subject S%02d: %d windows (%d FoG, %d NoFoG)",
                     sid, len(subjects_data[sid]["labels"]), n_fog,
                     len(subjects_data[sid]["labels"]) - n_fog)

    return subjects_data

In [ ]:
print("=" * 90)
print("  DAPHNET FoG DETECTION \u2014 IMPROVED PIPELINE")
print("=" * 90)
print(f"  Window: {WINDOW_SEC}s ({WINDOW_SAMPLES} samples)")
print(f"  Bandpass: {BP_LOW}-{BP_HIGH} Hz")
print(f"  Label threshold: {LABEL_THRESH} (majority voting)")
print(f"  Feature selection: top {K_FEATURES} (mutual information)")
print(f"  Threshold optimisation: Youden's J")
print()

t0 = time.time()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Load & preprocess
subjects_data = load_and_preprocess()

## Feature Extraction

In [ ]:
def extract_features_for_subject(windows: np.ndarray, fs: int = FS) -> pd.DataFrame:
    """Extract features from all windows of a subject."""
    extractor = FeatureExtractor(
        sampling_rate=fs,
        extract_time=True,
        extract_frequency=True,
        extract_wavelet=True,
        extract_nonlinear=False,  # too slow for pipeline
    )
    channel_groups = {
        "ankle": [0, 1, 2],
        "thigh": [3, 4, 5],
        "trunk": [6, 7, 8],
    }
    # The windows already include 3 magnitude channels at indices 9,10,11
    # FeatureExtractor will process all 12 channels via extract_from_window
    rows = []
    for i in range(len(windows)):
        feats = extractor.extract_from_window(windows[i], include_magnitude=False,
                                               channel_groups=None)
        rows.append(feats)
    return pd.DataFrame(rows)


def extract_all_features(subjects_data: Dict) -> Dict[int, Dict]:
    """Extract features for all subjects in parallel."""
    log.info("Extracting features for %d subjects (parallel)...", len(subjects_data))

    def _extract(sid):
        feat_df = extract_features_for_subject(subjects_data[sid]["windows"])
        return sid, feat_df

    results = Parallel(n_jobs=-1, verbose=0)(
        delayed(_extract)(sid) for sid in tqdm(subjects_data.keys(), desc="Feature extraction")
    )

    features = {}
    for sid, feat_df in results:
        features[sid] = {
            "X": feat_df,
            "y": subjects_data[sid]["labels"],
        }
        log.info("  S%02d: %d windows, %d features", sid, len(feat_df), feat_df.shape[1])
    return features

In [ ]:
# Step 2: Extract features
features = extract_all_features(subjects_data)